In [1]:
"""
Данный файл берет на вход csv файлы полученные через jupyter/stage3_squad_data_preparation.ipynb
И тестирует работу rag системы с выбранной моделью (которая уже запущена на 127.0.0.1:8081)
Логи запуска хранятся в log/model_answers
Графики можно построить через benchmark/bench/squad_summary.py или через вызов функции bench.squad_summary.py
"""


%cd ..
import ast
from functools import partial
from benchmark import bench

import pandas as pd

/home/kalisto/Documents/9_sem/BigDataAndML/Lossy-RAG


In [2]:
questions = pd.read_csv("data/bge-large-en-v1.5_Qwen3-Reranker-0.6B_30_3/questions_with_chunks.csv")
questions_paraphrased = pd.read_csv("data/bge-large-en-v1.5_Qwen3-Reranker-0.6B_30_3/questions_paraphrased_with_chunks.csv")

for col in ["answer_text", "answer_start_pos"]:
    questions[col] = questions[col].apply(ast.literal_eval)
    questions_paraphrased[col] = questions_paraphrased[col].apply(ast.literal_eval)

In [3]:
questions

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,88027,88027,valid,False,18914,Where was the practice place the Panthers used...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,88051,88051,valid,False,18916,What color were the footballs handed out to a ...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,88112,88112,valid,False,18920,What is the name of the service that gets loca...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,88115,88115,valid,False,18921,What is the name of the Host Committee's chari...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"[""Some NPOs which are particularly well known,..."
4,88144,88144,valid,False,18923,How many broadcast television partners does th...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,"[""In the United States, the game was televised..."
...,...,...,...,...,...,...,...,...,...,...,...,...
495,98112,98112,valid,False,20945,What is needed to pack electrons densely toget...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['When the electrons in a material are densely...
496,98116,98116,valid,False,20946,What is the force between nucleons?,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,"['However, a residual of the force is observed..."
497,98131,98131,valid,False,20949,What is the final effect of adding more and mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['Adding 100 to the program counter would caus...
498,98133,98133,valid,False,20950,What did Newton's mechanics affect?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


In [4]:
questions_paraphrased

,qid,orig_qid,role,is_paraphrase,pid,question,answer_text,answer_start_pos,title,context,n_chunks,chunks
0,100417,88027,valid,True,18914,Where did the Panthers train ahead of the Supe...,"[San Jose, the San Jose State practice facilit...","[22, 18, 22]",Super_Bowl_50,The Panthers used the San Jose State practice ...,1,['The Panthers used the San Jose State practic...
1,100463,88051,valid,True,18916,What hue were the balls given to many secondar...,"[gold, Gold]","[8, 357]",Super_Bowl_50,Various gold-themed promotions and initiatives...,1,['Various gold-themed promotions and initiativ...
2,100173,88112,valid,True,18920,Which program connects area firms with deals t...,[Business Connect],[164],Super_Bowl_50,"For the first time, the Super Bowl 50 Host Com...",1,"['For the first time, the Super Bowl 50 Host C..."
3,100518,88115,valid,True,18921,What is the Host Committee's philanthropic pro...,"[the 50 fund, 50 fund]","[200, 204]",Super_Bowl_50,The Super Bowl 50 Host Committee has vowed to ...,1,"[""Northwestern students also participate in th..."
4,100467,88144,valid,True,18923,With how many TV networks does the NFL have ai...,[three],[84],Super_Bowl_50,"In the United States, the game was televised b...",3,"[""In the United States, the game was televised..."
...,...,...,...,...,...,...,...,...,...,...,...,...
495,100330,98112,valid,True,20945,What is required to squeeze negatively charged...,"[energy, energy]","[579, 539]",Force,It is a common misconception to ascribe the st...,4,['When the electrons in a material are densely...
496,100287,98116,valid,True,20946,What interaction acts between protons and neut...,"[nuclear, nuclear force., nuclear force]","[216, 216, 216]",Force,The strong force only acts directly upon eleme...,4,['The strong and weak forces are nuclear force...
497,100582,98131,valid,True,20949,What ultimately results from attaching ever mo...,"[conservation of mechanical energy, the tensio...","[997, 623, 627]",Force,Tension forces can be modeled using ideal stri...,6,['Tension forces can be modeled using ideal st...
498,100398,98133,valid,True,20950,To what objects did the physics of Newton apply?,"[idealized point particles, idealized point pa...","[100, 100]",Force,Newton's laws and Newtonian mechanics in gener...,3,"[""Newton's laws and Newtonian mechanics in gen..."


In [ ]:

model_name = "Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS"
question_num = 50

gen = partial(
    bench.opeanai_model_generate_single_answer,
    model_name=model_name,
    max_new_tokens=4096,
    reasoning_budget_tokens=3072,
    do_sample=True,
    temperature=1,
    top_p=0.95,
    top_k=20,
)

In [6]:
score, elapsed = bench.run_squad(gen, questions.head(question_num), responses_path=f"log/model_answers/squad_no_context_{model_name}.jsonl")
score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), responses_path=f"log/model_answers/squad_p_no_context_{model_name}.jsonl")

print("─" * 80)
print(f"Model {model_name} score without context", "Num questions", question_num)
print("Score:", score, "\n", "Score paraphrase", score_p)
print("─" * 80)

squad_no_context_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

squad_p_no_context_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

────────────────────────────────────────────────────────────────────────────────
Model Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS score without context Num questions 50
Score: 28.0 
 Score paraphrase 22.0
────────────────────────────────────────────────────────────────────────────────


In [7]:
score, elapsed = bench.run_squad(gen, questions.head(question_num), context_column="context", responses_path=f"log/model_answers/squad_full_context_{model_name}.jsonl")
score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), context_column="context", responses_path=f"log/model_answers/squad_p_full_context_{model_name}.jsonl")

print("─" * 80)
print(f"Model {model_name} score with full context", "Num questions", question_num)
print("Score:", score, "\n", "Score paraphrase", score_p)
print("─" * 80)

squad_full_context_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

squad_p_full_context_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

────────────────────────────────────────────────────────────────────────────────
Model Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS score with full context Num questions 50
Score: 98.0 
 Score paraphrase 100.0
────────────────────────────────────────────────────────────────────────────────


In [8]:
#BAAI/bge-large-en-v1.5, k=30, reranker k=3

score, elapsed = bench.run_squad(gen, questions.head(question_num), context_column="chunks", responses_path=f"log/model_answers/squad_rag_{model_name}.jsonl")
score_p, elapsed_p = bench.run_squad(gen, questions_paraphrased.head(question_num), context_column="chunks", responses_path=f"log/model_answers/squad_p_rag_{model_name}.jsonl")

print("─" * 80)
print(f"Model {model_name} score with RAG context", "Num questions", question_num)
print("Score:", score, "\n", "Score paraphrase", score_p)
print("─" * 80)

squad_rag_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

squad_p_rag_Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS:   0%|          | 0/50 [00:00<?, ?req/s]

────────────────────────────────────────────────────────────────────────────────
Model Qwen3.8-Flash-Next-GSQ-RCO-IQ2_XS score with RAG context Num questions 50
Score: 72.0 
 Score paraphrase 68.0
────────────────────────────────────────────────────────────────────────────────
